# Using biotrainer autoeval for plm evaluation (zeroshot)

This notebook shows an example how to use the biotrainer `autoeval` module for automatic plm evaluation (zeroshot). We use the [ProteinGym DMS Supervised](https://marks.hms.harvard.edu/proteingym/ProteinGym_v1.3/DMS_ProteinGym_substitutions.zip) datasets that include various *deep mutational scanning* (DMS) fitness scores for protein mutations.

In [ ]:
# Install biotrainer if you haven't
# !pip install biotrainer

## Default Use Case: Model Download from Huggingface

The most convient option to use the biotrainer autoeval pipeline is to use the huggingface id of your plm. This will automatically download the model, calculate the embeddings and run the evaluation.

**Currently, this only works for [ProtBert](https://huggingface.co/Rostlab/prot_bert) and the [ESM-2](https://huggingface.co/facebook/esm2_t36_3B_UR50D) family. If you want to use other models, please use the advanced options below or add your model [here](../../biotrainer/bioengineer/bioengineer_models.py).**

In [ ]:
# Define variables
from biotrainer.autoeval import autoeval_pipeline
from biotrainer.bioengineer import ZeroShotMethod

embedder_name = "Rostlab/prot_bert"  # Replace with your plm's huggingface id. For alternatives, see "Advanced options" below
framework = "pgym"  # ProteinGym framework
zero_shot_method = ZeroShotMethod.MASKED_MARGINALS  # Replace with the zeroshot method you want to use

In [ ]:
# Run the pipeline
current_progress = None
for progress in autoeval_pipeline(embedder_name=embedder_name,
                                  framework=framework,
                                  zero_shot_method=zero_shot_method):
    print(progress)  # The pipeline is a generator function to inform the user about the current progress.
    current_progress = progress

In [ ]:
# Let's look at the results
final_report = None
if current_progress is None or current_progress.final_report is None:
    print("No results found.")  # Something went wrong
else:
    final_report = current_progress.final_report
    final_report.summary()

**Note that due to randomness in bootstrapping, results between `autoeval` and the [ProteinGym Leaderboard](https://proteingym.org/benchmarks) can differ. In our experiments, we found deviations from `+/- 0.001 to 0.025`.**

## Using a custom model

If you are running biotrainer-autoeval directly after training your model, the model will probably not be available on huggingface, but locally. Therefore, you can run the pipeline with a custom model implementation that allows you to use your model directly.

**This is showcased here for the ProtBert model**. The implementation uses the `CustomBioEngineerModel` interface, which allows you to define your own model implementation in a simplified way. If you need more control over the model, you can also extend the [BioEngineerModelWrapper](../../biotrainer/bioengineer/bioengineer_interfaces.py) interface.

In [ ]:
import re
import torch

from biotrainer.utilities import STANDARD_AAS
from transformers import BertTokenizer, BertForMaskedLM
from typing import List, Optional, Tuple, Iterable, Dict
from biotrainer.bioengineer import CustomBioEngineerModel

class CustomProtBertEngineer(CustomBioEngineerModel):
    def __init__(self, device: torch.device):
        super().__init__()
        # See https://huggingface.co/Rostlab/prot_bert for more details
        self.tokenizer = BertTokenizer.from_pretrained("Rostlab/prot_bert",
                                                       do_lower_case=False)
        self.model = BertForMaskedLM.from_pretrained("Rostlab/prot_bert").to(device).eval()
        self._device = device

    def get_name(self) -> str:
        return "CustomProtBertEngineer"

    def supported_methods(self) -> List[ZeroShotMethod]:
        return [ZeroShotMethod.WT_MARGINALS, ZeroShotMethod.MASKED_MARGINALS, ZeroShotMethod.PSEUDOPERPLEXITY]

    def run_model(self, input_ids: torch.Tensor, attention_mask: Optional[torch.Tensor] = None) -> torch.Tensor:
        with torch.no_grad():
            output = self.model(
                input_ids=input_ids,
                attention_mask=attention_mask,
            )
            logits = output.logits
            logits = logits[0]  # [1, seq_len, vocab_size] -> [seq_len, vocab_size]
        return logits

    def strip_special_tokens(self, tensor: torch.Tensor) -> torch.Tensor:
        return tensor[1:-1]

    def tokenize(self, batch: List[str]) -> Tuple[torch.Tensor, torch.Tensor]:
        enc = self.tokenizer(
            batch,
            add_special_tokens=True,
            is_split_into_words=False,
            padding="longest",
            return_tensors="pt",
        )
        tokenized_sequences = enc["input_ids"].to(self._device)
        attention_mask = enc["attention_mask"].to(self._device)
        return tokenized_sequences, attention_mask

    def get_mask_token_id(self) -> Optional[int]:
        return self.tokenizer.mask_token_id

    def preprocess_sequences(self, sequences: Iterable[str]) -> List[str]:
        preprocessed_seqs = []
        for seq in sequences:
            seq = " ".join(seq)  # Add whitespaces
            seq = re.sub(r"[UZOB]", "X", seq)  # Clean
            preprocessed_seqs.append(seq)
        return preprocessed_seqs

    def aa_to_idx(self) -> Dict[str, int]:
        return {aa: self.tokenizer.get_vocab()[aa] for aa in STANDARD_AAS}

In [ ]:
# Now we can create the custom model and a bio_engineer instance
from biotrainer.bioengineer import BioEngineer

device = torch.device("cuda")
bio_engineer = BioEngineer.from_custom_model(CustomProtBertEngineer(device=device), device=device)

In [ ]:
# We now use this bio_engineer object to run the autoeval zeroshot pipeline
current_progress = None
for progress in autoeval_pipeline(embedder_name="CustomProtBertEngineer",
                                  framework=framework,
                                  zero_shot_method=zero_shot_method,
                                  custom_bioengineer=bio_engineer):
    print(progress)  # The pipeline is a generator function to inform the user about the current progress.
    current_progress = progress

In [ ]:
# Let's look at the results and compare them
final_report_custom = current_progress.final_report
final_report_custom.summary()

# Compare to HuggingFace model
final_report_custom.compare([final_report], plot=True)